In [0]:
import os
import requests
from datetime import date, timedelta
from pyspark.sql import functions as F

base = "/Volumes/workspace/default/volume/"

spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.bronze")

DataFrame[]

In [0]:
arquivos = {
    "movies_info_TMDB_IMDB.csv": "tb_movies_info",
    "movies_financials_IMDB_TMDB.csv": "tb_movies_financials",
    "movies_metrics_IMDB_TMDB.csv": "tb_movies_metrics",
    "credits_and_tags_IMDB_TMDB.csv": "tb_credits_and_tags",
    "movies_reviews.csv": "tb_movies_reviews",
}

for arq, tabela in arquivos.items():
    df = (
        spark.read
        .option("header", True)
        .option("multiLine", True)  
        .option("escape", '"')
        .csv(os.path.join(base, arq))  
        .withColumn("ingestion_datetime", F.current_timestamp()) 
    )

    df.write.format("delta").mode("append").saveAsTable(f"workspace.bronze.{tabela}")
    print(f"OK: {tabela}")

OK: tb_movies_info
OK: tb_movies_financials
OK: tb_movies_metrics
OK: tb_credits_and_tags
OK: tb_movies_reviews


In [0]:
dbutils.widgets.text("data_inicio", (date.today() - timedelta(days=7)).strftime("%m-%d-%Y"))
dbutils.widgets.text("data_fim", date.today().strftime("%m-%d-%Y"))

data_inicio_formatada = dbutils.widgets.get("data_inicio")
data_fim_formatada = dbutils.widgets.get("data_fim")

url = (
    "https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/"
    "CotacaoDolarPeriodo(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)"
    f"?@dataInicial='{data_inicio_formatada}'&@dataFinalCotacao='{data_fim_formatada}'"
    "&$select=dataHoraCotacao,cotacaoCompra&$format=json"
)

resp = requests.get(url, timeout=30)
resp.raise_for_status()
dados = resp.json()["value"]

df_cot = (
    spark.createDataFrame(dados)
    .withColumn("ingestion_datetime", F.current_timestamp())
)

df_cot.write.format("delta").mode("append").saveAsTable("workspace.bronze.tb_cotacao_dolar")
print(f"OK: tb_cotacao_dolar ({df_cot.count()} linhas)")

OK: tb_cotacao_dolar (6 linhas)


In [0]:
for t in ["tb_movies_info", "tb_movies_financials", "tb_movies_metrics",
          "tb_credits_and_tags", "tb_movies_reviews", "tb_cotacao_dolar"]:
    print(t, spark.table(f"workspace.bronze.{t}").count())

display(spark.table("workspace.bronze.tb_movies_info").limit(5))

tb_movies_info 319788
tb_movies_financials 318495
tb_movies_metrics 206144
tb_credits_and_tags 211216
tb_movies_reviews 64824
tb_cotacao_dolar 12


id,tconst,title,original_title,original_language,release_date,runtime,status,overview,tagline,ingestion_datetime
293660,tt1431045,Deadpool,Deadpool,en,2016-02-09,108,Released,"The origin story of former Special Forces operative turned mercenary Wade Wilson, who, after being subjected to a rogue experiment that leaves him with accelerated healing powers, adopts the alter ego Deadpool. Armed with his new abilities and a dark, twisted sense of humor, Deadpool hunts down the man who nearly destroyed his life.",Witness the beginning of a happy ending.,2026-10-07T18:08:56.353Z
299536,tt4154756,AVENGERS: INFINITY WAR,Avengers: Infinity War,en,04-25-2018,149,Released,"As the Avengers and their allies have continued to protect the world from threats too large for any one hero to handle, a new danger has emerged from the cosmic shadows: Thanos. A despot of intergalactic infamy, his goal is to collect all six Infinity Stones, artifacts of unimaginable power, and use them to inflict his twisted will on all of reality. Everything the Avengers have fought for has led up to this moment - the fate of Earth and existence itself has never been more uncertain.",An entire universe. Once and for all.,2026-10-07T18:08:56.353Z
299534,tt4154796,Avengers: Endgame,Avengers: Endgame,en,2019-04-24,181,released,"After the devastating events of Avengers: Infinity War, the universe is in ruins due to the efforts of the Mad Titan, Thanos. With the help of remaining allies, the Avengers must assemble once more in order to undo Thanos' actions and restore order to the universe once and for all, no matter what consequences may be in store.",Avenge the fallen.,2026-10-07T18:08:56.353Z
475557,tt7286456,Joker,Joker,en,2019-10-01,122,Released,"During the 1980s, a failed stand-up comedian is driven insane and turns to a life of crime and chaos in Gotham City while becoming an infamous psychopathic crime figure.",Put on a happy face.,2026-10-07T18:08:56.353Z
271110,tt3498820,Captain America: Civil War,Captain America: Civil War,en,2016-04-27,147,Released,"Following the events of Age of Ultron, the collective governments of the world pass an act designed to regulate all superhuman activity. This polarizes opinion amongst the Avengers, causing two factions to side with Iron Man or Captain America, which causes an epic battle between former allies.",United we stand. Divided we fall.,2026-10-07T18:08:56.353Z
